# Day 6: Model Evaluation & Confidence Threshold Tuning

This notebook:
1. Loads the fine-tuned Legal-BERT model from Google Drive
2. Runs it on the held-out test set
3. Shows per-category F1, precision, recall for all 41 clause types
4. Finds the best confidence threshold per category
5. Saves an improved config your API can use later

**Run cells top to bottom, in order.**

## Cell 1: Install dependencies

In [ ]:
!pip install -q transformers datasets scikit-learn accelerate

## Cell 2: Mount Google Drive and set paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

DATA_DIR        = '/content/drive/MyDrive/contract-intelligence'
MODEL_DIR       = '/content/drive/MyDrive/contract-intelligence/model_output/final_model'
OUTPUT_DIR      = '/content/drive/MyDrive/contract-intelligence/evaluation'

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Confirm model files exist
print('Checking model files...')
for fname in ['config.json', 'model.safetensors', 'categories.json']:
    path = os.path.join(MODEL_DIR, fname)
    exists = os.path.exists(path)
    print(f'  {fname}: {"OK" if exists else "MISSING"}')

print('\nChecking data files...')
for fname in ['test.jsonl', 'categories.json']:
    path = os.path.join(DATA_DIR, fname)
    exists = os.path.exists(path)
    print(f'  {fname}: {"OK" if exists else "MISSING"}')

## Cell 3: Load model, tokenizer and categories

In [ ]:
import json
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

print('Loading categories...')
with open(os.path.join(MODEL_DIR, 'categories.json')) as f:
    categories = json.load(f)
num_labels = len(categories)
print(f'  {num_labels} clause categories loaded')

print('Loading tokenizer and model from Drive...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR)
model.eval()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
print(f'  Model loaded on: {device}')

## Cell 4: Load test set and run predictions

In [ ]:
from torch.utils.data import DataLoader, Dataset

MAX_LENGTH = 256
BATCH_SIZE = 32

class ContractDataset(Dataset):
    def __init__(self, jsonl_path):
        self.examples = []
        with open(jsonl_path) as f:
            for line in f:
                self.examples.append(json.loads(line))

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        return self.examples[idx]

def collate(batch):
    texts  = [ex['text'] for ex in batch]
    labels = [ex['labels'] for ex in batch]
    enc = tokenizer(texts, truncation=True, max_length=MAX_LENGTH,
                    padding=True, return_tensors='pt')
    enc['labels'] = torch.tensor(labels, dtype=torch.float)
    return enc

print('Loading test set...')
test_ds = ContractDataset(os.path.join(DATA_DIR, 'test.jsonl'))
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, collate_fn=collate)
print(f'  {len(test_ds)} test examples')

print('Running predictions (this takes a few minutes)...')
all_probs  = []
all_labels = []

with torch.no_grad():
    for i, batch in enumerate(test_loader):
        labels = batch.pop('labels').cpu().numpy()
        batch  = {k: v.to(device) for k, v in batch.items()}
        logits = model(**batch).logits
        probs  = torch.sigmoid(logits).cpu().numpy()
        all_probs.append(probs)
        all_labels.append(labels)
        if (i + 1) % 10 == 0:
            print(f'  Batch {i+1}/{len(test_loader)}')

all_probs  = np.vstack(all_probs)   # shape: (n_examples, 41)
all_labels = np.vstack(all_labels)  # shape: (n_examples, 41)
print(f'\nPredictions complete. Shape: {all_probs.shape}')

## Cell 5: Per-category evaluation at default threshold (0.5)

In [ ]:
from sklearn.metrics import f1_score, precision_score, recall_score

DEFAULT_THRESHOLD = 0.5

preds = (all_probs >= DEFAULT_THRESHOLD).astype(int)
labels_int = all_labels.astype(int)

print(f'Per-category results at threshold={DEFAULT_THRESHOLD}')
print(f'{"Category":<45} {"F1":>6} {"Prec":>6} {"Rec":>6} {"Support":>8}')
print('-' * 75)

results = []
for i, cat in enumerate(categories):
    support = int(labels_int[:, i].sum())
    if support == 0:
        f1 = prec = rec = 0.0
    else:
        f1   = f1_score(labels_int[:, i], preds[:, i], zero_division=0)
        prec = precision_score(labels_int[:, i], preds[:, i], zero_division=0)
        rec  = recall_score(labels_int[:, i], preds[:, i], zero_division=0)
    results.append({'category': cat, 'f1': f1, 'precision': prec,
                    'recall': rec, 'support': support})
    print(f'{cat:<45} {f1:>6.3f} {prec:>6.3f} {rec:>6.3f} {support:>8}')

# Overall
overall_f1   = f1_score(labels_int, preds, average='micro', zero_division=0)
overall_prec = precision_score(labels_int, preds, average='micro', zero_division=0)
overall_rec  = recall_score(labels_int, preds, average='micro', zero_division=0)
print('-' * 75)
print(f'{"OVERALL (micro)":<45} {overall_f1:>6.3f} {overall_prec:>6.3f} {overall_rec:>6.3f}')

# Flag weak categories
weak = [r for r in results if r['f1'] < 0.3 and r['support'] > 0]
print(f'\nCategories with F1 < 0.3 ({len(weak)} found):')
for r in sorted(weak, key=lambda x: x['f1']):
    print(f"  {r['category']:<45} F1={r['f1']:.3f}  support={r['support']}")

## Cell 6: Find the best threshold per category

Instead of using 0.5 for every category, we try thresholds from 0.1 to 0.9
and pick whichever gives the best F1 for that specific category.
This is the "heuristic improvement" step from your timetable.

In [ ]:
THRESHOLDS_TO_TRY = np.arange(0.1, 0.95, 0.05)

print('Finding best threshold per category...')
print(f'{"Category":<45} {"Best Thresh":>12} {"Best F1":>8} {"Default F1":>10}')
print('-' * 80)

best_thresholds = {}
improved = 0

for i, cat in enumerate(categories):
    support = int(labels_int[:, i].sum())
    default_f1 = results[i]['f1']

    if support == 0:
        best_thresholds[cat] = DEFAULT_THRESHOLD
        continue

    best_t, best_f1 = DEFAULT_THRESHOLD, default_f1
    for t in THRESHOLDS_TO_TRY:
        p = (all_probs[:, i] >= t).astype(int)
        f = f1_score(labels_int[:, i], p, zero_division=0)
        if f > best_f1:
            best_f1, best_t = f, t

    best_thresholds[cat] = float(round(best_t, 2))
    if best_t != DEFAULT_THRESHOLD:
        improved += 1

    marker = ' <-- improved' if best_t != DEFAULT_THRESHOLD else ''
    print(f'{cat:<45} {best_t:>12.2f} {best_f1:>8.3f} {default_f1:>10.3f}{marker}')

print(f'\n{improved} categories improved with a custom threshold.')

## Cell 7: Overall score with tuned thresholds

In [ ]:
# Apply best threshold per category
tuned_preds = np.zeros_like(preds)
for i, cat in enumerate(categories):
    tuned_preds[:, i] = (all_probs[:, i] >= best_thresholds[cat]).astype(int)

tuned_f1   = f1_score(labels_int, tuned_preds, average='micro', zero_division=0)
tuned_prec = precision_score(labels_int, tuned_preds, average='micro', zero_division=0)
tuned_rec  = recall_score(labels_int, tuned_preds, average='micro', zero_division=0)

print('=' * 50)
print('BEFORE threshold tuning (default 0.5):')
print(f'  F1 Micro  : {overall_f1:.4f}')
print(f'  Precision : {overall_prec:.4f}')
print(f'  Recall    : {overall_rec:.4f}')
print()
print('AFTER threshold tuning (per-category):')
print(f'  F1 Micro  : {tuned_f1:.4f}')
print(f'  Precision : {tuned_prec:.4f}')
print(f'  Recall    : {tuned_rec:.4f}')
print('=' * 50)

## Cell 8: Save evaluation results and threshold config to Drive

In [ ]:
# Save per-category results
eval_path = os.path.join(OUTPUT_DIR, 'per_category_results.json')
with open(eval_path, 'w') as f:
    json.dump({
        'overall_default_threshold': {
            'f1_micro': overall_f1,
            'precision_micro': overall_prec,
            'recall_micro': overall_rec,
        },
        'overall_tuned_threshold': {
            'f1_micro': tuned_f1,
            'precision_micro': tuned_prec,
            'recall_micro': tuned_rec,
        },
        'per_category': results,
    }, f, indent=2)
print(f'Evaluation results saved to: {eval_path}')

# Save threshold config -- this is what the FastAPI app will load later
# to decide when to flag a clause as present
threshold_path = os.path.join(OUTPUT_DIR, 'thresholds.json')
with open(threshold_path, 'w') as f:
    json.dump(best_thresholds, f, indent=2)
print(f'Threshold config saved to:   {threshold_path}')

print('\nDay 6 complete.')
print('These two files will be used by the FastAPI app in Day 8.')